## Vickers Hardness to GPa Calculator

This calculator determines Vickers Hardness directly in GPa using precise physical constants and indenter geometry. The calculation employs:
- Gravitational acceleration: $g_n = 9.80665 \, \text{m/s}^2$
- Indenter geometry: $136^\circ$

### Core Formula

$$H_{GPa} = \frac{18.1848523 \cdot F_{kgf}}{1000 \cdot d^2}$$

### Variable Definitions

- $F_{kgf} = 1.0$ (Fixed Load in kilogram-force)
- $d = \frac{d_1 + d_2}{2}$ (Mean diagonal length in mm, where $d_1$ and $d_2$ are the two diagonals of the indentation)

In [229]:
# Vickers Hardness Calculator
# Fixed load parameter
F_kgf = 1

# Collect user input for diagonals
d1_input = input("Enter the first diagonal d1 (in μm): ")
d2_input = input("Enter the second diagonal d2 (in μm): ")

# Convert inputs to float and convert from μm to mm
d1 = float(d1_input) / 1000
d2 = float(d2_input) / 1000

# Calculate mean diagonal
d = (d1 + d2) / 2

# Safety check for division by zero
if d == 0:
    print("Error: Mean diagonal cannot be zero. Please provide non-zero diagonal values.")
else:
    # Apply the Vickers Hardness formula
    H_gpa = (18.1848523 * F_kgf) / (1000 * (d ** 2))
    
    # Display results
    print(f"\n--- Vickers Hardness Results ---")
    print(f"First diagonal (d1): {d1*1000:.4f} μm")
    print(f"Second diagonal (d2): {d2*1000:.4f} μm")
    print(f"Mean diagonal (d): {d*1000:.4f} μm")
    print(f"Vickers Hardness (H): {H_gpa:.4f} GPa")


--- Vickers Hardness Results ---
First diagonal (d1): 87.3520 μm
Second diagonal (d2): 87.5280 μm
Mean diagonal (d): 87.4400 μm
Vickers Hardness (H): 2.3784 GPa


## Plastic Zone Radius ($C_s$) Calculation

This section calculates the plastic zone radius generated by the Vickers indentation based on the Expanding Cavity Model. All length units are maintained in $\mu\text{m}$.

### Derivation Formulas

The plastic zone radius is derived through a series of calculated steps:

**Yield Stress:**
$$\sigma_{ys} = \frac{H}{2.57}$$

**Projected Area:**
$$A_c = \frac{d^2}{2}$$

**Equivalent Contact Radius:**
$$a_s = \sqrt{\frac{A_c}{\pi}}$$

**Indenter Shape Parameter:**
$$R = \frac{a_s}{0.635}$$

**Plastic Zone Depth:**
$$Z_{ys} = R \cdot \exp\left[ \frac{1}{2} \left( \frac{H}{ f*\sigma_{ys}} - \frac{2}{3} \right) \right] - 1.217 a_s$$

**Parameter $c$:**
$$c = Z_{ys} + 1.217 a_s$$

**Plastic Zone Radius at Surface:**
$$C_s = R \sqrt{\left(\frac{c}{R}\right)^2 - 0.5868}$$

In [230]:
import math

f = 1.079 # for vickers

# Convert mean diagonal from mm to μm for plastic zone calculations
d_mean = d * 1000  # d is already in mm from the previous step

# Step 1: Calculate Yield Stress
sigma_ys = H_gpa / 2.57

# Step 2: Calculate Projected Area (in μm²)
A_c = (d_mean ** 2) / 2

# Step 3: Calculate Equivalent Contact Radius (in μm)
a_s = math.sqrt(A_c / math.pi)

# Step 4: Calculate Indenter Shape Parameter (in μm)
R = a_s / 0.635

# Step 5: Calculate Plastic Zone Depth (in μm)
exponent = (1/2) * (H_gpa / (f * sigma_ys) - 2/3)
Z_ys = (R * math.exp(exponent)) - (1.217 * a_s)

# Step 6: Calculate Parameter c (in μm)
c = Z_ys + 1.217 * a_s

# Step 7: Calculate Plastic Zone Radius at Surface (in μm)
C_s = R * math.sqrt((c / R) ** 2 - 0.5868)

C_s_area = math.pi * (C_s ** 2)

# Print intermediate variables and results
print(f"\n--- Plastic Zone Radius Calculation (Expanding Cavity Model) ---")
print(f"Mean diagonal (d_mean): {d_mean:.3f} μm")
print(f"Hardness (H): {H_gpa:.3f} GPa")
print(f"\nIntermediate Variables:")
print(f"Yield Stress (σ_ys): {sigma_ys:.3f} GPa")
print(f"Projected Area (A_c): {A_c:.3f} μm²")
print(f"Equivalent Contact Radius (a_s): {a_s:.3f} μm")
print(f"Indenter Shape Parameter (R): {R:.3f} μm")
print(f"Plastic Zone Depth (Z_ys): {Z_ys:.3f} μm")
print(f"Parameter (c): {c:.3f} μm")
print(f"\n*** Plastic Zone Radius at Surface (C_s): {C_s:.3f} μm")
print(f"Plastic Zone Area at Surface (C_s_area): {C_s_area:.3f} μm²")


--- Plastic Zone Radius Calculation (Expanding Cavity Model) ---
Mean diagonal (d_mean): 87.440 μm
Hardness (H): 2.378 GPa

Intermediate Variables:
Yield Stress (σ_ys): 0.925 GPa
Projected Area (A_c): 3822.877 μm²
Equivalent Contact Radius (a_s): 34.884 μm
Indenter Shape Parameter (R): 54.935 μm
Plastic Zone Depth (Z_ys): 87.053 μm
Parameter (c): 129.506 μm

*** Plastic Zone Radius at Surface (C_s): 122.479 μm
Plastic Zone Area at Surface (C_s_area): 47127.035 μm²


In [231]:
import csv
import os

# Path to the CSV file
csv_path = os.path.join(os.path.dirname(os.getcwd()), "IM2PROP_data", "IM2PROP_dataset_homo_v1.csv")

# Let user enter the image ID
image_id = input("Enter the image ID (e.g., HO_32): ")

# Use calculated values from previous cells
hardness_value = round(H_gpa, 4)
plastic_zone_area = round(C_s_area, 3)

print(f"\n--- Values to append ---")
print(f"Image ID: {image_id}")
print(f"Hardness Value: {hardness_value}")
print(f"Plastic Zone Area: {plastic_zone_area}")

# Append the new row to the CSV
with open(csv_path, "a") as f:
    writer = csv.writer(f)
    writer.writerow([image_id, hardness_value, plastic_zone_area])

print(f"\nRow successfully appended to {csv_path}")


--- Values to append ---
Image ID: HO_94
Hardness Value: 2.3784
Plastic Zone Area: 47127.035

Row successfully appended to /Users/jaychou/Desktop/LBS/IM2PROP/IM2PROP_data/IM2PROP_dataset_homo_v1.csv
